In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# --- Load & prep (same as yours, but avoid dropping zeros) ---
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2023": "pop_2023",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989"
})

# Ensure numeric
for c in ["church_persistence_index", "income_1989_real_2023", "pct_hs_1990",
          "pop_2023", "firms_2022", "firms_1989"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Minimal NA drop (keep zeros!)
df = df.dropna(subset=[
    "church_persistence_index", "income_1989_real_2023", "pct_hs_1990",
    "pop_2023", "firms_2022", "firms_1989", "State"
])

# Exposure (must be > 0 to take log). Counties with pop<=0 are nonsensical; drop if any.
df = df[df["pop_2023"] > 0]

# Controls: standardize income & education (as in your OLS)
scaler = StandardScaler()
df[["income_1989_scaled", "pct_hs_1990_scaled"]] = scaler.fit_transform(
    df[["income_1989_real_2023", "pct_hs_1990"]]
)

# Historical firms: use log1p to avoid dropping zero-1989 counties
df["log1p_firms_1989"] = np.log1p(df["firms_1989"])

# --- Main PPML with exposure and State FE ---
# E[firms_2022 | X] = exp( β0 + βX*X + ... + δ_state + log(pop_2023) )
# -> offset = log(pop_2023) gives a *per-capita* rate model.
ppml = smf.glm(
    formula="firms_2022 ~ church_persistence_index + income_1989_scaled + pct_hs_1990_scaled + log1p_firms_1989 + C(State)",
    data=df,
    family=sm.families.Poisson(),
    offset=np.log(df["pop_2023"])
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["State"]}
)

print(ppml.summary())

# Optional: percent effect of a 1-unit change in persistence
beta = ppml.params["church_persistence_index"]
pct_change = (np.exp(beta) - 1) * 100
print(f"Approx. % change in firms per capita for +1 unit persistence: {pct_change:.2f}%")


                 Generalized Linear Model Regression Results                  
Dep. Variable:             firms_2022   No. Observations:                 2978
Model:                            GLM   Df Residuals:                     2926
Model Family:                 Poisson   Df Model:                           51
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -26704.
Date:                Thu, 21 Aug 2025   Deviance:                       35751.
Time:                        14:38:27   Pearson chi2:                 3.92e+04
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               